<a href="https://colab.research.google.com/github/Durgvanshi/Learning-Deep-Learning/blob/main/VisionTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
# import libs
import torch
import torchvision
import torchvision.transforms as transforms
import torch.utils.data as dataloader
import torch.nn as nn

In [42]:
# transforming image from PIL format
PIL_to_tensor_transformation = transforms.Compose([transforms.ToTensor()])


In [43]:
# import dataset
train_dataset = torchvision.datasets.MNIST(root= './data' ,train=True , download=True , transform = PIL_to_tensor_transformation);
val_dataset = torchvision.datasets.MNIST(root='./data' , train=False , download=True , transform = PIL_to_tensor_transformation)

In [44]:
# variables
num_classes = 10
batch_size = 64
num_channels = 1
img_size = 28
patch_size = 7
num_patches = (img_size//patch_size) ** 2
embedding_dimension = 64
attention_heads = 4
transformer_blocks = 4
learning_rate = 0.001
epochs = 5
mlp_hidden_nodes = 128


In [45]:
#define batches

train_loader = dataloader.DataLoader(train_dataset , batch_size = batch_size , shuffle= True)
val_loader = dataloader.DataLoader(val_dataset , batch_size = batch_size , shuffle= True)

In [46]:

# Patch Embedding

class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embed = nn.Conv2d(num_channels , embedding_dimension, kernel_size = 7 , stride= patch_size)

  def forward(self , x):
    x = self.patch_embed(x)
    x = x.flatten(2)
    x = x.transpose(1,2)

    return x

    #Todo [class]
    #Todo positional embedding


In [47]:
# Transformer encoder

class TransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dimension)
    self.layer_norm2 = nn.LayerNorm(embedding_dimension)
    self.multihead_attention = nn.MultiheadAttention(embedding_dimension , attention_heads , batch_first = True)
    self.mlp = nn.Sequential(
        nn.Linear(embedding_dimension , mlp_hidden_nodes),
        nn.GELU(),
        nn.Linear(mlp_hidden_nodes , embedding_dimension)
    )

  def forward(self,x):
      residual1 = x
      x = self.layer_norm1(x)
      x = self.multihead_attention(x,x,x)[0]
      x = x + residual1
      residual2 = x
      x = self.layer_norm2(x)
      x = self.mlp(x)
      x = x + residual2

      return x



In [48]:
# MLP_head for classification

class MLP_head(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dimension)
    self.mlp_head = nn.Linear(embedding_dimension , num_classes)

  def forward(self , x):
      x = self.layer_norm1(x)
      x = self.mlp_head(x)

      return x

In [49]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = PatchEmbedding()
    self.cls_token = nn.Parameter(torch.randn(1,1,embedding_dimension))
    self.position_embedding = nn.Parameter(torch.randn(1,num_patches+1 ,embedding_dimension))
    self.transformer_blocks = nn.Sequential(*[TransformerEncoder() for _ in range(transformer_blocks)])
    self.mlp_head = MLP_head()

  def forward(self , x):
      x = self.patch_embedding(x)
      B = x.size(0)
      class_tokens = self.cls_token.expand(B , -1 ,-1)
      x = torch.cat((class_tokens , x ), dim =1)
      x = self.transformer_blocks(x)
      x = x[:,0]
      x = self.mlp_head(x)

      return x

In [50]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
criterion = nn.CrossEntropyLoss()

In [53]:
# training

for epoch in range(epochs):
  model.train()
  total_loss = 0
  correct_epoch = 0
  total_epoch = 0
  print(f"\nEpoch {epoch + 1}/{epochs}")

  for batch_idx , (images , labels) in enumerate(train_loader):
    images , labels = images.to(device) , labels.to(device)

    optimizer.zero_grad()
    output = model(images)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    preds = output.argmax(dim=1)
    correct = (preds == labels).sum().item()

    correct_epoch += correct
    total_epoch += labels.size(0)

    if batch_idx % 100 == 0:
      accuracy = 100.0 * correct / labels.size(0)
      print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f} | Batch Acc: {accuracy:.2f}%")

  epoch_acc = 100.0 * correct_epoch / total_epoch
  avg_loss = total_loss / len(train_loader)
  print(f"Summary Epoch {epoch + 1}: Avg Loss: {avg_loss:.4f} | Epoch Acc: {epoch_acc:.2f}%")


Epoch 1/5
Batch 0/938 | Loss: 0.3969 | Batch Acc: 87.50%
Batch 100/938 | Loss: 0.2859 | Batch Acc: 89.06%
Batch 200/938 | Loss: 0.1416 | Batch Acc: 95.31%
Batch 300/938 | Loss: 0.4376 | Batch Acc: 87.50%
Batch 400/938 | Loss: 0.1778 | Batch Acc: 93.75%
Batch 500/938 | Loss: 0.1994 | Batch Acc: 93.75%
Batch 600/938 | Loss: 0.2770 | Batch Acc: 93.75%
Batch 700/938 | Loss: 0.0822 | Batch Acc: 96.88%
Batch 800/938 | Loss: 0.0544 | Batch Acc: 96.88%
Batch 900/938 | Loss: 0.0526 | Batch Acc: 98.44%
Summary Epoch 1: Avg Loss: 0.1409 | Epoch Acc: 95.41%

Epoch 2/5
Batch 0/938 | Loss: 0.0903 | Batch Acc: 96.88%
Batch 100/938 | Loss: 0.0639 | Batch Acc: 96.88%
Batch 200/938 | Loss: 0.2232 | Batch Acc: 93.75%
Batch 300/938 | Loss: 0.0905 | Batch Acc: 95.31%
Batch 400/938 | Loss: 0.1734 | Batch Acc: 90.62%
Batch 500/938 | Loss: 0.1727 | Batch Acc: 95.31%
Batch 600/938 | Loss: 0.1104 | Batch Acc: 98.44%
Batch 700/938 | Loss: 0.0239 | Batch Acc: 100.00%
Batch 800/938 | Loss: 0.1094 | Batch Acc: 96.

In [54]:

# Validation summary
model.eval()
val_loss = 0
correct_val = 0
total_val = 0

with torch.no_grad():
  for images, labels in val_loader:
    images, labels = images.to(device), labels.to(device)

    output = model(images)
    preds = output.argmax(dim=1)
    correct_val += (preds == labels).sum().item()
    loss = criterion(output, labels)
    total_val += labels.size(0)



val_accuracy = 100.0 * correct_val / total_val

print(f"--- Validation Summary ---")
print(f"Accuracy: {val_accuracy:.2f}%")

--- Validation Summary ---
Accuracy: 95.27%
